# Arkenstone ARK-020 — Multi-Skill Continual Cognition Generalization

Runs the full ARK-020 campaign on a **T4 GPU**: four sequentially acquired skills
(relational binding x2, successor-rule induction, inverse retrieval) under 7 arms
including reactive vs predictive vs hybrid Guardians, with a capability registry and
risk-allocated replay. Exact-resumable across Colab sessions; rerun the same notebook
until the final bundle appears. ~140k updates total; the runtime calibration cell
reports the expected session count without changing the protocol.

Run cells top to bottom. Do not modify pinned commit or thresholds after seeing results.

In [ ]:
import os, shutil, subprocess, sys, torch
PINNED_RUNNER_COMMIT = '59a85a614265e35905bdb2453a5f54f48e95b8a7'
REPO = '/content/An-Ra-the-new-AGI-ark020'
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU before running.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','50','--branch','Arkenstone','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',REPO], check=True)
subprocess.run(['git','-C',REPO,'checkout','--detach',PINNED_RUNNER_COMMIT], check=True)
head = subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'], text=True).strip()
assert head == PINNED_RUNNER_COMMIT, (head, PINNED_RUNNER_COMMIT)
paths = [
 'experiments/ARK-020/ark020_core.py',
 'experiments/ARK-020/run_ark020.py',
 'tests/test_ark020.py',
 'experiments/ARK-019/ark019_v4_core.py',
 'experiments/ARK-019/run_ark019_v4.py',
 'experiments/ARK-019/run_ark019_v3.py',
 'experiments/ARK-018/ark018_v3_common.py',
 'experiments/ARK-018/ark018_v3_binding_fast.py',
]
for p in paths:
    subprocess.run([sys.executable,'-m','py_compile',os.path.join(REPO,p)], check=True)
print('compile gate: PASS on', len(paths), 'files')
r = subprocess.run([sys.executable,'-m','unittest','tests.test_ark020'], cwd=REPO)
assert r.returncode == 0, 'pure test suite failed'
print('pure tests: PASS')
print('Environment ready. The next cell mounts Drive and runs the campaign;')
print('the runner self-gates: substrate identity, 48-token gate, parents, dose,')
print('exact-resume smoke, runtime calibration, then matched sets.')

In [ ]:
# Full campaign. 225-minute session timebox per run; rerun this cell to resume.
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
runner = os.path.join(REPO,'experiments/ARK-020/run_ark020.py')
env = dict(os.environ); env['PYTHONUNBUFFERED']='1'
proc = subprocess.run([sys.executable, runner, '--mode', 'all'], cwd=REPO, env=env)
print('CAMPAIGN RETURN CODE:', proc.returncode)
if proc.returncode != 0:
    print('A failure receipt and partial JSONs are in the PARTIAL zip. Rerun this cell to resume exactly.')

In [ ]:
from pathlib import Path
root = Path('/content/drive/MyDrive/genisis-arkenstone/ARK020_CONTINUAL_V1')
print('Result files:')
for p in sorted(root.glob('*.json')):
    print(' ', p.name, p.stat().st_size, 'bytes')
res = root / 'ARK-020_RESULT.json'
if res.exists():
    print()
    print('ARK-020 RESULT:')
    print(res.read_text())
import subprocess
z = root / 'ARKENSTONE_ARK020_CONTINUAL_RESULTS.zip'
if not z.exists():
    z = root / 'ARKENSTONE_ARK020_CONTINUAL_PARTIAL.zip'
if z.exists():
    print()
    print('ZIP:', z, z.stat().st_size, 'bytes')
    try:
        from google.colab import files
        files.download(str(z))
    except Exception as exc:
        print('Manual download: Files panel > drive > MyDrive > genisis-arkenstone > ARK020_CONTINUAL_V1', exc)